# BIOT 6900 Module 1 Updated
### Computational Environment Setup & Tool Ecosystem

**Name:** Bhuvanesh Karthik Nilavalagan  



## Part A — Environment verification

In [1]:
import sys, platform
print("Python version:   ", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Platform:         ", platform.platform())

Python version:    3.11.16
Python executable: /opt/anaconda3/envs/biot6900/bin/python
Platform:          macOS-15.3-arm64-arm-64bit


In [2]:
import os
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "not detected"))

Conda environment: biot6900


In [3]:
!git --version

git version 2.39.5 (Apple Git-154)


In [4]:
import Bio, pandas as pd, numpy as np, requests
print("BioPython:", Bio.__version__)
print("pandas:   ", pd.__version__)
print("numpy:    ", np.__version__)
print("requests: ", requests.__version__)

BioPython: 1.88
pandas:    2.3.3
numpy:     2.4.6
requests:  2.34.2


**Checkpoint A:** Python 3.11 is running from the `biot6900` environment, Git is installed, and
BioPython, pandas, numpy and requests all import cleanly. Everything below depends on these, so
any later error is a problem with the query or the data rather than the setup.

## Part B — BioPython quickstart

In [5]:
from Bio.Seq import Seq
dna = Seq("ATGGCCATTGTAATGGGCCGCTGA")
print("Length:", len(dna))
print("Protein:", dna.translate())

Length: 24
Protein: MAIVMGR*


**TODO (B):** `.translate()` split the 24-base sequence into 8 codons and converted each to the
amino acid it encodes, giving `MAIVMGR*`. The `*` is the TGA stop codon. BioPython starts at
position 0 rather than searching for a start codon, so the reading frame is whatever the input
implies.

## Part C — Biomedical database API exploration

### C1 — PubMed (via `Bio.Entrez`)

In [6]:
from Bio import Entrez
Entrez.email = "karthik.b@northeastern.edu"   # required by NCBI
handle = Entrez.esearch(db="pubmed", term="AlphaFold protein structure", retmax=5)
record = Entrez.read(handle)
print("Total hits:", record["Count"])
print("PMIDs:", record["IdList"])

Total hits: 3046
PMIDs: ['42943733', '42943275', '42943032', '42941728', '42939705']


**TODO (C1) — Note:** Searched `AlphaFold protein structure` in PubMed with `retmax=5`, so five
PMIDs came back out of the much larger total hit count. `esearch` returns identifiers only; a
second `esummary` or `efetch` call is needed for titles and abstracts.

For target discovery this is the first filter. Before committing to a target you check what is
already published about it, and the PMID is the stable handle that links a computational result
back to the evidence behind it.

### C2 — UniProt (REST API)

In [7]:
import requests
r = requests.get("https://rest.uniprot.org/uniprotkb/P04637.json")
data = r.json()
name = data["proteinDescription"]["recommendedName"]["fullName"]["value"]
print("Accession:", "P04637")
print("Protein:", name)
print("Gene:", data["genes"][0]["geneName"]["value"])
print("Length:", data["sequence"]["length"], "residues")

Accession: P04637
Protein: Cellular tumor antigen p53
Gene: TP53
Length: 393 residues


**TODO (C2) — Note:** Accession `P04637` returned Cellular tumor antigen p53, gene `TP53`, 393
residues. The endpoint gives the full UniProtKB record as JSON, so name, gene, organism and the
sequence itself all come from one request.

For target discovery UniProt is where a gene name becomes a protein you could actually drug. The
sequence retrieved here is what you would feed to a structure prediction tool, and the accession
is what cross-references this protein to the PDB entry queried next.

### C3 — PDB (RCSB REST API)

In [8]:
import requests
r = requests.get("https://data.rcsb.org/rest/v1/core/entry/1TUP")
entry = r.json()
print("Title:", entry["struct"]["title"])
print("Method:", entry["exptl"][0]["method"])
print("Resolution:", entry["rcsb_entry_info"]["resolution_combined"][0], "A")
# the structure file itself:
# https://files.rcsb.org/download/1TUP.pdb

Title: TUMOR SUPPRESSOR P53 COMPLEXED WITH DNA
Method: X-RAY DIFFRACTION
Resolution: 2.2 A


**TODO (C3) — Note:** `1TUP` is the p53 core domain complexed with DNA, solved by X-ray
crystallography. This is the same protein as C2, now as an experimental 3D structure rather than
a sequence.

For target discovery this is the druggability check. Sequence tells you the parts list; structure
tells you whether there is a pocket a small molecule could bind. The resolution matters because it
sets how much confidence to place in the atomic positions when docking against them.

### C4 — GWAS Catalog (EBI REST API v2)

In [9]:
import requests
from collections import Counter

# NOTE: the v1 endpoint in the lab guide now returns HTTP 410 Gone — the GWAS
# Catalog retired REST API v1 in August 2026. v2 applies filters directly to
# the resource (?rs_id=...) instead of using /singleNucleotidePolymorphisms/.
base = "https://www.ebi.ac.uk/gwas/rest/api/v2"
r = requests.get(f"{base}/associations", params={"rs_id": "rs7412", "size": 50})
payload = r.json()

records = payload["_embedded"]["associations"]
print("Variant: rs7412")
print("Total associations:", payload["page"]["totalElements"])

strongest = min(records, key=lambda a: a["p_value"])
print("Strongest in page:", strongest["reported_trait"][0], "| p =", strongest["p_value"])

traits = Counter(t["efo_trait"] for a in records for t in a["efo_traits"])
print("Distinct traits in page:", len(traits))
for trait, n in traits.most_common(5):
    print(f"  {n}x {trait}")

Variant: rs7412
Total associations: 1183
Strongest in page: triglyceride (minimum, inv-norm transformed) | p = 1e-323
Distinct traits in page: 17
  8x total cholesterol measurement
  8x red cell distribution width
  7x hypercholesterolemia
  4x triglyceride measurement
  4x hyperlipidemia


**TODO (C4) — Note:** Looked up `rs7412`, one of the two SNPs defining the APOE e2/e3/e4
haplotypes. The v1 endpoint from the lab guide returns HTTP 410 because the Catalog retired that
API in August 2026, so this uses v2, where searches are filters on the resource rather than
separate `findBy` endpoints.

The Catalog holds 1,183 associations for this one variant, and the 50 retrieved map to 17 distinct
traits, mostly lipid measurements. The strongest reached p = 1e-323, which is the floating-point
floor rather than a real value.

For target discovery GWAS is the human-genetic evidence that a gene actually matters in disease,
which is the strongest predictor of a target surviving clinical trials. The caveat is that an
association flags a locus, not a causal gene, which is why it has to be read alongside the protein
and structural layers above.

## Part D — Download & load sample multi-omics data

Three layers, three public repositories, downloaded straight into `shared/Data/` so the notebook
is reproducible without manual file handling.

In [10]:
from pathlib import Path
import requests

# Put downloads in shared/Data at the repo root, wherever this notebook runs from.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), Path.cwd())
DATA_DIR = root / "shared" / "Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Data directory:", DATA_DIR)

Data directory: /Users/bhavadharanin/Desktop/biot6900/shared/Data


### D1 — Transcriptomics (GEO)

In [11]:
import gzip, io, pandas as pd

# GSE2034: breast cancer tumour expression, Wang et al. 2005.
url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE2nnn/GSE2034/matrix/GSE2034_series_matrix.txt.gz"
out = DATA_DIR / "GSE2034_expression.tsv"

if not out.exists():
    raw = gzip.decompress(requests.get(url, timeout=120).content).decode("utf-8", "replace")
    # the expression table starts after the !series_matrix_table_begin marker
    table = raw.split("!series_matrix_table_begin\n")[1].split("!series_matrix_table_end")[0]
    out.write_text(table)

geo = pd.read_csv(out, sep="\t", index_col=0)
print("GEO GSE2034 shape:", geo.shape)
geo.iloc[:3, :4]

GEO GSE2034 shape: (22283, 286)


,GSM36777,GSM36778,GSM36779,GSM36780
ID_REF,,,,
1007_s_at,3848.1,6520.9,5285.7,4043.7
1053_at,228.9,112.5,178.4,398.7
117_at,213.1,189.8,269.7,312.4


### D2 — Expression across tissues (GTEx)

In [12]:
import pandas as pd

# Median expression of TP53 across GTEx tissues (same protein as C2 and C3).
url = "https://gtexportal.org/api/v2/expression/medianGeneExpression"
r = requests.get(url, params={"gencodeId": "ENSG00000141510.18", "itemsPerPage": 100}, timeout=60)
r.raise_for_status()

gtex = pd.DataFrame(r.json()["data"])
gtex.to_csv(DATA_DIR / "gtex_TP53_median_expression.tsv", sep="\t", index=False)
print("GTEx shape:", gtex.shape)
gtex[["tissueSiteDetailId", "median", "unit"]].sort_values("median", ascending=False).head()

GTEx shape: (54, 7)


,tissueSiteDetailId,median,unit
22,Cells_EBV-transformed_lymphocytes,77.4845,TPM
45,Skin_Sun_Exposed_Lower_leg,37.4491,TPM
44,Skin_Not_Sun_Exposed_Suprapubic,35.3415,TPM
21,Cells_Cultured_fibroblasts,33.3962,TPM
40,Ovary,33.0791,TPM


### D3 — Cancer multi-omics with clinical annotation (cBioPortal)

In [13]:
import pandas as pd

# TP53 mutations in the TCGA breast cancer study.
url = ("https://www.cbioportal.org/api/molecular-profiles/"
       "brca_tcga_pan_can_atlas_2018_mutations/mutations")
r = requests.get(url, params={"sampleListId": "brca_tcga_pan_can_atlas_2018_all",
                              "entrezGeneId": 7157,            # TP53
                              "projection": "DETAILED"}, timeout=120)
r.raise_for_status()

cbio = pd.json_normalize(r.json())
cbio.to_csv(DATA_DIR / "cbioportal_brca_TP53_mutations.tsv", sep="\t", index=False)
print("cBioPortal shape:", cbio.shape)
cbio[["sampleId", "proteinChange", "mutationType"]].head()

cBioPortal shape: (353, 30)


,sampleId,proteinChange,mutationType
0,TCGA-3C-AALI-01,S183*,Nonsense_Mutation
1,TCGA-A1-A0SI-01,R175H,Missense_Mutation
2,TCGA-A1-A0SK-01,M133K,Missense_Mutation
3,TCGA-A1-A0SO-01,Y220C,Missense_Mutation
4,TCGA-A1-A0SP-01,S183*,Nonsense_Mutation


In [14]:
# Summary of all three datasets
import pandas as pd
pd.DataFrame([
    {"source": "GEO GSE2034",  "layer": "transcriptomics", "rows": geo.shape[0],  "columns": geo.shape[1]},
    {"source": "GTEx TP53",    "layer": "expression",      "rows": gtex.shape[0], "columns": gtex.shape[1]},
    {"source": "cBioPortal",   "layer": "genomics",        "rows": cbio.shape[0], "columns": cbio.shape[1]},
])

,source,layer,rows,columns
0,GEO GSE2034,transcriptomics,22283,286
1,GTEx TP53,expression,54,7
2,cBioPortal,genomics,353,30


**TODO (D) — Note:** Three datasets, one per layer. **GEO GSE2034** is tumour transcriptomics,
rows are microarray probes and columns are breast tumour samples, so each value is a normalised
expression level. **GTEx** gives median TP53 expression across human tissues, which is the
normal-tissue baseline rather than a disease measurement. **cBioPortal** returns somatic TP53
mutations in TCGA breast tumours, one row per mutation call with the amino acid change. Shapes
are printed above.

Why this matters for target discovery: these three answer different questions about the same
gene. cBioPortal says whether it is mutated in disease, GEO says whether its expression changes
in tumours, and GTEx says where it is normally expressed, which is the main predictor of
off-target toxicity. A target that looks good in tumour data but is highly expressed in heart or
brain is a bad target, and only the normal-tissue layer reveals that. TP53 is used throughout so
the three layers describe the same protein queried in C2 and C3.